In [ ]:
!apt update && apt install -y openslide-tools
!pip install openslide-python

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd '/content/drive/My Drive/2_PHDWork/1_Objective-I/8_ScalingErrorEstimation/'

In [ ]:
from openslide import OpenSlide
import os
import matplotlib.pyplot as plt
import numpy as np
import cv2 as cv

from scipy.spatial import KDTree
import pickle

from shapely import get_coordinates, concave_hull, buffer, LineString, Point
from shapely.geometry import Polygon
from shapely.affinity import translate
from shapely.ops import unary_union
from shapely.geometry.multipolygon import MultiPolygon

In [ ]:
"""
  Find the contour points of the high resolution mask and the corresponding
  downsampled mask.
"""
def find_contour_points(mskwsi,s):
  """
    Step-1: Read the HR mask and calculate the original mask boundary points.
  """
  h_msk = np.array(mskwsi.read_region((0,0),0,mskwsi.level_dimensions[0]))[:,:,0]
  h_msk[h_msk > 0] = 255
  h_contours, _ = cv.findContours(h_msk,cv.RETR_EXTERNAL,cv.CHAIN_APPROX_NONE)
  h_contours = sorted(h_contours, key=cv.contourArea, reverse=True)

  """
    Step-2: Downscale the mask to the required scale and the obtained mask is
            considered as LR mask
  """
  width, height = mskwsi.level_dimensions[0]
  n_width, n_height = int(width/s), int(height/s)
  l_msk = cv.resize(h_msk,(n_width,n_height),interpolation=cv.INTER_AREA)

  """
    Step-3: Determine the mask boundary points of LR mask
  """
  l_contours, _ = cv.findContours(l_msk,cv.RETR_EXTERNAL,cv.CHAIN_APPROX_NONE)
  l_contours = sorted(l_contours, key=cv.contourArea, reverse=True)

  """
    Step-4: Scaling the LR boundary points to represent the scaled co-ordinates.
  """
  s_contours = []
  for cnt in l_contours:
    s_cnt = cnt.copy().astype(np.float32)
    s_cnt[:,:,0] = s_cnt[:,:,0] * s
    s_cnt[:,:,1] = s_cnt[:,:,1] * s
    s_cnt = (np.floor(s_cnt)).astype(np.int32)
    s_contours.append(s_cnt)

  """
    Step-5: Interpolating the scaled boundary points.
  """
  h_msk = np.zeros_like(h_msk, dtype=np.uint8)
  cv.drawContours(h_msk,s_contours,-1,255,-1)
  s_contours, _ = cv.findContours(h_msk,cv.RETR_EXTERNAL,cv.CHAIN_APPROX_NONE)
  s_contours = sorted(s_contours, key=cv.contourArea, reverse=True)

  """
    Step-6: Find the matching contours.
  """
  h_contours_s,s_contours = find_matching_contours(h_contours,s_contours)

  """
    Step-7: Find the proposed method based contours and the matching contours.
  """
  h_size = mskwsi.level_dimensions[0]
  p_contours = find_proposed_method_based_contours(l_contours,s,(h_size[1],h_size[0]))
  h_contours_p,p_contours = find_matching_contours(h_contours,p_contours)

  return h_contours_s, s_contours, h_contours_p, p_contours

def compute_centroid(contour):
  contour = np.asarray(contour).reshape(-1, 2)
  return np.mean(contour, axis=0)

def find_matching_contours(h_contours, s_contours):
  # Compute centroids
  centroids_high = np.array([compute_centroid(c) for c in h_contours])
  centroids_low = np.array([compute_centroid(c) for c in s_contours])

  n_h_contours = []
  n_s_contours = []

  for i, c_low in enumerate(centroids_low):
    # Compute distances to all high-res centroids
    distances = np.linalg.norm(centroids_high - c_low, axis=1)
    idx = np.argmin(distances)

    n_h_contours.append(h_contours[idx])
    n_s_contours.append(s_contours[i])

  return n_h_contours, n_s_contours

def find_proposed_method_based_contours(o_contours,s,h_size):
  new_points = []
  par = 0.5

  for i in range(len(o_contours)):
    cnt = np.reshape(o_contours[i].copy(),(-1,2)).astype(np.float32)

    if(cnt.shape[0] <= 10):
      if(cnt.shape[0] == 1):
        poly1 = buffer(Point(cnt), 0.3,join_style='mitre')
      else:
        poly1 = buffer(LineString(cnt.tolist()), 0.3,join_style='mitre')

      comb_ply = scale_polygon_coordinates(poly1,s,s)
      if not comb_ply.is_valid:
        comb_ply = comb_ply.buffer(0.1,join_style='mitre')
    else:
      poly1 = Polygon(cnt)
      poly2 = translate(poly1, xoff=par, yoff=par)    # Bottom-right
      poly3 = translate(poly1, xoff=-par, yoff=par)   # Bottom-left
      poly4 = translate(poly1, xoff=par, yoff=-par)   # Top-right

      poly1 = scale_polygon_coordinates(poly1,s,s)
      poly2 = scale_polygon_coordinates(poly2,s,s)
      poly3 = scale_polygon_coordinates(poly3,s,s)
      poly4 = scale_polygon_coordinates(poly4,s,s)

      if not poly1.is_valid:
        poly1 = poly1.buffer(0.1,join_style='mitre')
      if not poly2.is_valid:
        poly2 = poly2.buffer(0.1,join_style='mitre')
      if not poly3.is_valid:
        poly3 = poly3.buffer(0.1,join_style='mitre')
      if not poly4.is_valid:
        poly4 = poly4.buffer(0.1,join_style='mitre')

      comb_ply = unary_union([poly1,poly2,poly3,poly4])

      if isinstance(comb_ply, MultiPolygon):
        comb_ply = concave_hull(comb_ply,ratio=0.001)

    new_points.append(np.reshape(np.array(list(comb_ply.exterior.coords)),(-1,1,2)))

  new_points = convert_to_displayable_contours(new_points,h_size[0],h_size[1])
  bin_mask = np.zeros(h_size,dtype=np.uint8)
  bin_mask = cv.fillPoly(bin_mask, new_points, (255))
  new_points, _ = cv.findContours(bin_mask,cv.RETR_EXTERNAL,cv.CHAIN_APPROX_NONE)
  new_points = sorted(new_points, key=cv.contourArea, reverse=True)

  return new_points

def scale_polygon_coordinates(polygon,s_r,s_c):
  coordinates = get_coordinates(polygon)
  coordinates[:,0] *= s_c
  coordinates[:,1] *= s_r

  s_polygon = Polygon(coordinates)

  return s_polygon

def convert_to_displayable_contours(contours,max_r,max_c):
  dis_cont = []
  for i in range(len(contours)):
    cnt = np.ceil(contours[i].copy()).astype(np.int32)
    cnt[:,:,0] = np.clip(cnt[:,:,0],0,max_c)
    cnt[:,:,1] = np.clip(cnt[:,:,1],0,max_r)
    dis_cont.append(cnt)

  return dis_cont

In [ ]:
"""
  The error of the original boundary points with respect to scaled boundary
  points are estimated.
"""
def get_errors_between_curves(h_contours,s_contours):
  length = len(h_contours)
  error_values = []
  quandrant_values = []
  for i in range(length):
    curve_B = h_contours[i].squeeze(axis=1) # Original
    curve_A = s_contours[i].squeeze(axis=1) # Scaled
    signed_distances = signed_distance_curve_to_curve(curve_A, curve_B)
    error_values.append(signed_distances)
    quadrant = classify_quadrants_with_respect_to_centroid(curve_A)
    quandrant_values.append(quadrant)

  return [error_values,quandrant_values]

def signed_distance_curve_to_curve(curve_A, curve_B):
  tree_B = KDTree(curve_B)
  signed_distances = []

  for pt in curve_A:
    dist, idx = tree_B.query(pt)
    nearest = curve_B[idx]

    # Compute normal vector at nearest point on curve B
    prev_idx = (idx - 1) % len(curve_B)
    next_idx = (idx + 1) % len(curve_B)
    tangent = curve_B[next_idx] - curve_B[prev_idx]
    normal = np.array([-tangent[1], tangent[0]]).astype(np.float64)
    norm = np.linalg.norm(normal)
    if norm > 1e-8:
      normal /= norm
    else:
      # fallback: skip this point or assign default normal
      normal = np.array([0.0, 0.0])

    # Signed distance: dot product with normal
    vec = pt - nearest
    euclidean_dist = np.linalg.norm(vec) # Euclidean distance

    """
      Positive means outside curve-B
      Negative means inside curve-B
    """
    signed_dist = np.dot(vec, normal)
    signed_distances.append(np.sign(signed_dist) * euclidean_dist)

  return np.array(signed_distances)


def classify_quadrants_with_respect_to_centroid(boundary_points):
  """
    Classify each boundary point into one of four quadrants relative to the centroid:
    0 - Top Right
    1 - Top Left
    2 - Bottom Left
    3 - Bottom Right

    Parameters:
    - boundary_points: (N, 2) array of (x, y) coordinates

    Returns:
    - quadrant_labels: (N,) array of integers in {0, 1, 2, 3}
  """
  boundary_points = np.asarray(boundary_points)
  centroid = np.mean(boundary_points, axis=0)

  dx = boundary_points[:, 0] - centroid[0]
  dy = boundary_points[:, 1] - centroid[1]

  # Top is smaller y (image coordinates)
  quadrant_labels = np.zeros(len(boundary_points), dtype=int)

  quadrant_labels[(dx < 0) & (dy < 0)] = 1  # Top Left
  quadrant_labels[(dx < 0) & (dy >= 0)] = 2 # Bottom Left
  quadrant_labels[(dx >= 0) & (dy >= 0)] = 3 # Bottom Right
  # Top Right is default (0)

  return quadrant_labels

In [ ]:
def save_error_values(destpath,filename,s,error_values):
  destdir = "Scale_"+str(s)
  destpath = os.path.join(destpath,destdir)
  if not os.path.exists(destpath):
    os.makedirs(destpath)

  name, ext = os.path.splitext(filename)
  save_file = os.path.join(destpath,name+".pkl")
  with open(save_file, "wb") as f:
    pickle.dump(error_values, f)

  # # For testing
  # with open(save_file, "rb") as f:
  #   loaded = pickle.load(f)

  # print(error_values[0].shape)
  # print(loaded[0].shape)

  return

In [ ]:
"""
  For visualization of the ground-truth and scaled curve.
"""
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
def generate_visualization(mskwsi,curves_A, curves_B):
  # zoom_region = (x_min, x_max, y_min, y_max)
  zoom_region = [(1500, 3000, 4000, 5000)]
  width, height = mskwsi.level_dimensions[0]
  image_size = (height, width)  # (height, width)

  visualize_signed_distances_multi(curves_A, curves_B, zoom_region=zoom_region)
  return

def visualize_signed_distances_multi(curves_A, curves_B, zoom_region=None):
  fig, ax2 = plt.subplots(1, 1)
  plot_both_curves(curves_A, curves_B, ax2)
  ax2.set_title('To be updated')
  ax2.set_xlim(left=0)
  ax2.set_ylim(bottom=0)
  ax2.invert_yaxis()
  ax2.grid(linestyle='--')
  ax2.legend(fontsize="10")

  # Create inset axes for zoomed
  position = 3
  if zoom_region:
    for region in zoom_region:
      x1, x2, y1, y2 = region  # ROI bounds
      axins = inset_axes(ax2, width="40%", height="40%", loc=position)
      plot_both_curves(curves_A, curves_B, axins)
      axins.set_xlim(x1, x2)
      axins.set_ylim(y2, y1)  # inverted y-axis
      axins.invert_yaxis()
      axins.set_xticks([])
      axins.set_yticks([])
      axins.set_title('Zoomed view', fontsize=10)
      mark_inset(ax2, axins, loc1=2, loc2=4, fc="none", ec="0.5")
      position += 1

  fig.set_figheight(15)
  fig.set_figwidth(10)
  fig.tight_layout()
  # file_name = "./Results/New_output.pdf"
  # fig.savefig(file_name,format='pdf',bbox_inches='tight',pad_inches = 0)
  plt.show()

  return

def plot_both_curves(curve_A, curve_B, axes):
  index = 0
  for cntr in curve_A:
    cntr = np.reshape(cntr,(-1,2))
    if(index == 0):
      plot_contour(cntr,axes,'#0000ff','Ground truth')
      index += 1
    else:
      plot_contour(cntr,axes,'#0000ff',None)

  index = 0
  for cntr in curve_B:
    cntr = np.reshape(cntr,(-1,2))
    if(index == 0):
      plot_contour(cntr,axes,'#ff0000','Simple scaling')
      index += 1
    else:
      plot_contour(cntr,axes,'#ff0000',None)

  return

def plot_contour(cntr,axes,colr,labl):
  x = cntr[:,0]
  y = cntr[:,1]
  if(labl != None):
    axes.plot(x,y,color=colr,linewidth=0.5,label=labl)
  else:
    axes.plot(x,y,color=colr,linewidth=0.5)
  return

In [ ]:
def entry_fun(maskpath,destpath):
  if os.path.isfile(maskpath):
    files_names = [os.path.basename(maskpath)]
    maskpath = os.path.dirname(maskpath)
  elif os.path.isdir(maskpath):
    files_names = os.listdir(maskpath)
  else:
    print("Path does not exist or is something else.")
    files_names = []

  skip_list = ['006f6aa35a78965c92fffd1fbd53a058_mask.tiff',
               '00ca0c23961b5510be46c917be11c43e_mask.tiff']

  if(len(files_names) > 0):
    index = 0
    while(index < len(files_names)):
      file_name = files_names[index]
      if(file_name in skip_list):
        index += 1
        continue
      print(f'\t{index}: Processing the WSI file {file_name}')
      wsiname = os.path.join(maskpath,file_name)
      mskwsi = OpenSlide(wsiname)
      t_level =  mskwsi.level_count-1
      s = mskwsi.level_downsamples[t_level] / 2

      """
        Step-1: Get the boundary points of the mask corresponding to the original
                and scaled masks
      """
      h_contours_s, s_contours, h_contours_p, p_contours = find_contour_points(mskwsi,s)

      """
         Step-2: Compute the error between the boundary curves (ground-truth and simplae scaled).
      """
      error_values_s = get_errors_between_curves(h_contours_s,s_contours)

      """
        Step-3: Compute the error between the boundary curves (ground-truth and proposed method).
      """
      error_values_p = get_errors_between_curves(h_contours_p,p_contours)

      total_error = []
      total_error.extend(error_values_s)
      total_error.extend(error_values_p)

      """
        Un-comment if you want to save the error values.
      """
      save_error_values(destpath,file_name,s,total_error)

      """
        Un-comment if you want the visualization.
      """
      # generate_visualization(mskwsi,h_contours, s_contours)


      mskwsi.close()
      index += 1
      # if(index == 1): # Uncomment this when you use train_label_masks/karolinska/
      #   break
  return

In [ ]:
def main():
  maskpath = [
               "../../2_Objective-II/0_Dataset/1_Panda/test_label_masks/radboud/"
             ]
  destpath = "./1_ErrorValues/1_Inter_Area/"

  pathindex = 0
  entry_fun(maskpath[pathindex],destpath)
  return

In [ ]:
main()